In [6]:
import requests
import json

ESPN_ID = "401859966"
url = f"https://site.web.api.espn.com/apis/site/v2/sports/basketball/nba/summary?event={ESPN_ID}"

r = requests.get(url)
data = r.json()

print("Status HTTP :", r.status_code)
print("Clés racine :", list(data.keys()))

Status HTTP : 200
Clés racine : ['boxscore', 'format', 'gameInfo', 'lastFiveGames', 'leaders', 'seasonseries', 'injuries', 'broadcasts', 'pickcenter', 'odds', 'againstTheSpread', 'header', 'videos', 'wallclockAvailable', 'predictor', 'winprobability', 'news', 'meta', 'ticketsInfo', 'standings']


In [7]:
odds = data.get('odds', [])

print("Type :", type(odds))
print("Nombre d'entrées :", len(odds))
print()
for i, o in enumerate(odds):
    print(f"--- Entrée {i} ---")
    print(json.dumps(o, indent=2))

Type : <class 'list'>
Nombre d'entrées : 0



In [8]:
pickcenter      = data.get('pickcenter', [])
predictor       = data.get('predictor', {})
against_spread  = data.get('againstTheSpread', [])

print("=== PICKCENTER ===")
print("Type :", type(pickcenter), "/ Nb entrées :", len(pickcenter))
for i, p in enumerate(pickcenter):
    print(f"--- Entrée {i} ---")
    print(json.dumps(p, indent=2))

print()
print("=== PREDICTOR ===")
print(json.dumps(predictor, indent=2))

print()
print("=== AGAINST THE SPREAD ===")
print("Type :", type(against_spread), "/ Nb entrées :", len(against_spread))
for i, a in enumerate(against_spread):
    print(f"--- Entrée {i} ---")
    print(json.dumps(a, indent=2))

=== PICKCENTER ===
Type : <class 'list'> / Nb entrées : 0

=== PREDICTOR ===
{
  "header": "Matchup Predictor",
  "homeTeam": {
    "id": "18",
    "gameProjection": "58.1",
    "teamChanceLoss": "41.9"
  },
  "awayTeam": {
    "id": "24",
    "gameProjection": "41.9",
    "teamChanceLoss": "58.1"
  }
}

=== AGAINST THE SPREAD ===
Type : <class 'list'> / Nb entrées : 2
--- Entrée 0 ---
{
  "team": {
    "id": "24",
    "uid": "s:40~l:46~t:24",
    "displayName": "San Antonio Spurs",
    "abbreviation": "SA",
    "links": [
      {
        "href": "https://www.espn.com/nba/team/_/name/sa/san-antonio-spurs",
        "text": "Clubhouse"
      },
      {
        "href": "https://www.espn.com/nba/team/schedule/_/name/sa",
        "text": "Schedule"
      }
    ],
    "logo": "https://a.espncdn.com/i/teamlogos/nba/500/sa.png",
    "logos": [
      {
        "href": "https://a.espncdn.com/i/teamlogos/nba/500/sa.png",
        "width": 500,
        "height": 500,
        "alt": "",
        "rel

In [9]:
pc = data['pickcenter'][0] if data.get('pickcenter') else None

if not pc:
    print("Pas de pickcenter disponible")
else:
    provider   = pc['provider']['name']
    spread     = pc.get('details', '–')          # ex: "NY -2.5"
    over_under = pc.get('overUnder', '–')        # ex: 215.5

    ml_home    = pc['homeTeamOdds'].get('moneyLine', '–')
    ml_away    = pc['awayTeamOdds'].get('moneyLine', '–')
    fav_home   = pc['homeTeamOdds'].get('favorite', False)

    spread_home_close = pc.get('pointSpread', {}).get('home', {}).get('close', {})
    spread_away_close = pc.get('pointSpread', {}).get('away', {}).get('close', {})
    spread_home_open  = pc.get('pointSpread', {}).get('home', {}).get('open', {})

    over_close = pc.get('total', {}).get('over', {}).get('close', {})
    under_close = pc.get('total', {}).get('under', {}).get('close', {})

    print(f"Provider       : {provider}")
    print(f"Spread         : {spread}")
    print(f"Over/Under     : {over_under}")
    print(f"ML domicile    : {ml_home} ({'favori' if fav_home else 'outsider'})")
    print(f"ML extérieur   : {ml_away}")
    print(f"Spread dom     : {spread_home_close.get('line','–')} @ {spread_home_close.get('odds','–')}")
    print(f"Spread ext     : {spread_away_close.get('line','–')} @ {spread_away_close.get('odds','–')}")
    print(f"Spread open dom: {spread_home_open.get('line','–')} (mouvement)")
    print(f"Over close     : {over_close.get('line','–')} @ {over_close.get('odds','–')}")
    print(f"Under close    : {under_close.get('line','–')} @ {under_close.get('odds','–')}")

    # Predictor (dans summary, différent de l'endpoint core API)
    pred = data.get('predictor', {})
    print()
    print(f"Predictor home : {pred.get('homeTeam', {}).get('gameProjection','–')}%")
    print(f"Predictor away : {pred.get('awayTeam', {}).get('gameProjection','–')}%")

Pas de pickcenter disponible


In [10]:
wp = data.get('winprobability', [])

print("Type :", type(wp), "/ Nb entrées :", len(wp))

if len(wp) > 0:
    print("\nPremière entrée :")
    print(json.dumps(wp[0], indent=2))
    print("\nDernière entrée :")
    print(json.dumps(wp[-1], indent=2))
else:
    print("Vide — probablement peuplé uniquement en live ou post-match")

Type : <class 'list'> / Nb entrées : 0
Vide — probablement peuplé uniquement en live ou post-match


In [11]:
API_KEY = "35c1429947235730065b330ebf17df13"
BASE_URL = "https://api.the-odds-api.com/v4"

r = requests.get(f"{BASE_URL}/sports", params={"apiKey": API_KEY})

print("Status HTTP :", r.status_code)
print("Requêtes restantes :", r.headers.get("x-requests-remaining"))
print("Requêtes utilisées :", r.headers.get("x-requests-used"))
print()

sports = r.json()
# Filtrer basketball uniquement
basket = [s for s in sports if "basket" in s.get("key", "").lower()]
print(f"Sports basket trouvés ({len(basket)}) :")
for s in basket:
    print(f"  - {s['key']} | {s['title']} | actif: {s.get('active')} | en cours: {s.get('has_outrights', '?')}")

Status HTTP : 200
Requêtes restantes : 500
Requêtes utilisées : 0

Sports basket trouvés (3) :
  - basketball_nba | NBA | actif: True | en cours: False
  - basketball_nba_championship_winner | NBA Championship Winner | actif: True | en cours: True
  - basketball_wnba | WNBA | actif: True | en cours: False


In [13]:
r = requests.get(f"{BASE_URL}/sports/basketball_nba/odds", params={
    "apiKey": API_KEY,
    "regions": "eu",
    "markets": "h2h,spreads,totals",
    "oddsFormat": "decimal",
})

print("Status HTTP :", r.status_code)
print("Requêtes restantes :", r.headers.get("x-requests-remaining"))
print()

matchs = r.json()
print(f"Nombre de matchs retournés : {len(matchs)}")
print()

# Tous les bookmakers uniques
tous_books = set()
for m in matchs:
    for b in m.get('bookmakers', []):
        tous_books.add(b['key'])

print(f"Bookmakers EU uniques trouvés ({len(tous_books)}) :")
for b in sorted(tous_books):
    print(f"  - {b}")

print()
print("=== Liste complète des matchs ===")
for m in matchs:
    books = [b['key'] for b in m.get('bookmakers', [])]
    print(f"{m.get('commence_time')} | {m.get('away_team')} @ {m.get('home_team')} | {len(books)} books : {books}")

Status HTTP : 200
Requêtes restantes : 497

Nombre de matchs retournés : 1

Bookmakers EU uniques trouvés (23) :
  - betanysports
  - betclic_fr
  - betfair_ex_eu
  - betonlineag
  - betsson
  - codere_it
  - coolbet
  - gtbets
  - leovegas_se
  - matchbook
  - mybookieag
  - nordicbet
  - onexbet
  - pinnacle
  - pmu_fr
  - sport888
  - tipico_de
  - unibet_fr
  - unibet_nl
  - unibet_se
  - williamhill
  - winamax_de
  - winamax_fr

=== Liste complète des matchs ===
2026-06-09T00:40:00Z | San Antonio Spurs @ New York Knicks | 23 books : ['unibet_nl', 'unibet_se', 'leovegas_se', 'pmu_fr', 'coolbet', 'sport888', 'betonlineag', 'unibet_fr', 'onexbet', 'codere_it', 'betclic_fr', 'nordicbet', 'mybookieag', 'tipico_de', 'betsson', 'pinnacle', 'winamax_fr', 'winamax_de', 'williamhill', 'gtbets', 'betfair_ex_eu', 'matchbook', 'betanysports']


In [14]:
m = matchs[0]
print(f"Match : {m['away_team']} @ {m['home_team']}")
print(f"Date  : {m['commence_time']}")
print()

books_cibles = ['betclic_fr', 'unibet_fr', 'winamax_fr', 'pmu_fr']

for b in m.get('bookmakers', []):
    if b['key'] not in books_cibles:
        continue
    print(f"=== {b['title']} ===")
    for market in b.get('markets', []):
        print(f"  Marché : {market['key']}")
        for outcome in market.get('outcomes', []):
            print(f"    {outcome['name']} : {outcome['price']}")
    print()

Match : San Antonio Spurs @ New York Knicks
Date  : 2026-06-09T00:40:00Z

=== PMU (FR) ===
  Marché : h2h
    New York Knicks : 1.78
    San Antonio Spurs : 2.1
  Marché : totals
    Over : 1.83
    Under : 1.81

=== Unibet (FR) ===
  Marché : h2h
    New York Knicks : 1.77
    San Antonio Spurs : 2.1

=== Betclic (FR) ===
  Marché : h2h
    New York Knicks : 1.77
    San Antonio Spurs : 2.08

=== Winamax (FR) ===
  Marché : h2h
    New York Knicks : 1.76
    San Antonio Spurs : 2.1



In [15]:
r = requests.get(f"{BASE_URL}/sports/basketball_nba/odds", params={
    "apiKey": API_KEY,
    "regions": "eu,us",
    "markets": "h2h,spreads,totals,alternate_spreads,alternate_totals,team_totals,player_props",
    "oddsFormat": "decimal",
})

print("Requêtes restantes :", r.headers.get("x-requests-remaining"))
print()

m = r.json()[0]
print(f"Match : {m['away_team']} @ {m['home_team']}")
print()

# Tous les marchés uniques par bookmaker
for b in m.get('bookmakers', []):
    marches = [mk['key'] for mk in b.get('markets', [])]
    if marches:
        print(f"{b['title']} : {marches}")

Requêtes restantes : 497



KeyError: 0

In [18]:
print("Status HTTP :", r.status_code)
print("Réponse brute :", r.json())

Status HTTP : 422
Réponse brute : {'message': 'Markets not supported by this endpoint: alternate_spreads, alternate_totals, team_totals', 'error_code': 'INVALID_MARKET', 'details_url': 'https://the-odds-api.com/liveapi/guides/v4/api-error-codes.html#invalid-market'}


In [17]:
r = requests.get(f"{BASE_URL}/sports/basketball_nba/odds", params={
    "apiKey": API_KEY,
    "regions": "eu,us",
    "markets": "h2h,spreads,totals,alternate_spreads,alternate_totals,team_totals",
    "oddsFormat": "decimal",
})

print("Status HTTP :", r.status_code)
print("Requêtes restantes :", r.headers.get("x-requests-remaining"))

m = r.json()[0]
print(f"\nMatch : {m['away_team']} @ {m['home_team']}")
print()

for b in m.get('bookmakers', []):
    marches = [mk['key'] for mk in b.get('markets', [])]
    if marches:
        print(f"{b['title']} : {marches}")

Status HTTP : 422
Requêtes restantes : 497


KeyError: 0

In [19]:
r = requests.get(f"{BASE_URL}/sports/basketball_nba/odds", params={
    "apiKey": API_KEY,
    "regions": "eu,us",
    "markets": "h2h,spreads,totals",
    "oddsFormat": "decimal",
})

print("Status HTTP :", r.status_code)
print("Requêtes restantes :", r.headers.get("x-requests-remaining"))

m = r.json()[0]
print(f"\nMatch : {m['away_team']} @ {m['home_team']}")
print()

for b in m.get('bookmakers', []):
    marches = [mk['key'] for mk in b.get('markets', [])]
    if marches:
        print(f"{b['title']} : {marches}")

Status HTTP : 200
Requêtes restantes : 491

Match : San Antonio Spurs @ New York Knicks

FanDuel : ['h2h', 'spreads', 'totals']
DraftKings : ['h2h', 'spreads', 'totals']
Unibet (NL) : ['h2h', 'spreads', 'totals']
Unibet (SE) : ['h2h', 'spreads']
LeoVegas (SE) : ['h2h', 'spreads']
BetRivers : ['h2h', 'spreads', 'totals']
PMU (FR) : ['h2h', 'totals']
Coolbet : ['h2h', 'spreads', 'totals']
888sport : ['h2h', 'spreads', 'totals']
BetOnline.ag : ['h2h', 'spreads', 'totals']
Unibet (FR) : ['h2h']
BetMGM : ['h2h', 'spreads', 'totals']
LowVig.ag : ['h2h', 'spreads', 'totals']
1xBet : ['h2h', 'spreads', 'totals']
Codere (IT) : ['h2h', 'totals']
Betclic (FR) : ['h2h']
Nordic Bet : ['h2h', 'totals']
MyBookie.ag : ['h2h', 'spreads', 'totals']
Tipico : ['h2h', 'spreads', 'totals']
Betsson : ['h2h', 'totals']
Bovada : ['h2h', 'spreads', 'totals']
Pinnacle : ['h2h', 'spreads', 'totals']
Winamax (FR) : ['h2h']
Winamax (DE) : ['h2h']
William Hill : ['h2h']
GTbets : ['h2h', 'spreads', 'totals']
Betfair 

In [20]:
m = r.json()[0]
print(f"Match : {m['away_team']} @ {m['home_team']}\n")

for b in m.get('bookmakers', []):
    print(f"=== {b['title']} ===")
    for market in b.get('markets', []):
        print(f"  [{market['key']}]")
        for outcome in market.get('outcomes', []):
            extra = f" | points: {outcome['point']}" if 'point' in outcome else ""
            print(f"    {outcome['name']} : {outcome['price']}{extra}")
    print()

Match : San Antonio Spurs @ New York Knicks

=== FanDuel ===
  [h2h]
    New York Knicks : 1.77
    San Antonio Spurs : 2.1
  [spreads]
    New York Knicks : 1.95 | points: -2.5
    San Antonio Spurs : 1.87 | points: 2.5
  [totals]
    Over : 1.95 | points: 216.5
    Under : 1.87 | points: 216.5

=== DraftKings ===
  [h2h]
    New York Knicks : 1.77
    San Antonio Spurs : 2.1
  [spreads]
    New York Knicks : 1.95 | points: -2.5
    San Antonio Spurs : 1.87 | points: 2.5
  [totals]
    Over : 1.87 | points: 215.5
    Under : 1.95 | points: 215.5

=== Unibet (NL) ===
  [h2h]
    New York Knicks : 1.78
    San Antonio Spurs : 2.1
  [spreads]
    New York Knicks : 1.89 | points: -2.0
    San Antonio Spurs : 1.92 | points: 2.0
  [totals]
    Over : 1.92 | points: 216.0
    Under : 1.89 | points: 216.0

=== Unibet (SE) ===
  [h2h]
    New York Knicks : 1.78
    San Antonio Spurs : 2.1
  [spreads]
    New York Knicks : 1.89 | points: -2.0
    San Antonio Spurs : 1.92 | points: 2.0

=== LeoV